<a href="https://colab.research.google.com/github/Milanjavoor/AI-Learning/blob/main/intrusiondetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder ,MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dropout,
    LSTM,
    Dense
)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix
df=pd.read_csv("/content/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
print(df.head())
df.columns = df.columns.str.strip()
print(df.columns)
df.replace([np.inf,-np.inf],np.nan,inplace=True)
df.dropna(inplace=True)
encoder=LabelEncoder()
df["label"] = encoder.fit_transform(
    df["Label"]
)
x = df.drop(columns=["Label", "label"])

y = df["label"]
scalar=MinMaxScaler()
x=scalar.fit_transform(x)
seq_length=10
x_seq=[]
y_seq=[]
for i in range(len(x)-seq_length):
  x_seq.append(x[i:i+seq_length])
  y_seq.append(y.iloc[i+seq_length])
x_seq=np.array(x_seq)
y_seq=np.array(y_seq)
x_train,x_test,y_train,y_test=train_test_split(
    x_seq,y_seq,
    test_size=0.2,
    random_state=42
)
model=Sequential()
model.add(
    LSTM(64,
         return_sequences=True,
         input_shape=(x_train.shape[1],x_train.shape[2])
         )
)
model.add(Dropout(0.3))
model.add(LSTM(32))
model.add(Dropout(0.3))
model.add(Dense(16, activation="relu"))
model.add(Dense(1,activation="sigmoid"))
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
earlystop=EarlyStopping(
    monitor="val_loss",
    restore_best_weights=True,
    patience=5

)
print(x_train.dtype)

print(y_train.dtype)

print(df.dtypes)

print(df.head())
history=model.fit(
    x_train,
    y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.3,
    callbacks=[earlystop]
)

loss,accuracy=model.evaluate(x_test,y_test)
print(accuracy)
predict=model.predict(x_test)
predict=(predict>0.5).astype(int)
cm=confusion_matrix(
    y_test,
    predict
)

print(cm)

    Destination Port   Flow Duration   Total Fwd Packets  \
0              54865               3                   2   
1              55054             109                   1   
2              55055              52                   1   
3              46236              34                   1   
4              54863               3                   2   

    Total Backward Packets  Total Length of Fwd Packets  \
0                        0                           12   
1                        1                            6   
2                        1                            6   
3                        1                            6   
4                        0                           12   

    Total Length of Bwd Packets   Fwd Packet Length Max  \
0                             0                       6   
1                             6                       6   
2                             6                       6   
3                             6                 

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


float64
int64
Destination Port                 int64
Flow Duration                    int64
Total Fwd Packets                int64
Total Backward Packets           int64
Total Length of Fwd Packets      int64
                                ...   
Idle Std                       float64
Idle Max                         int64
Idle Min                         int64
Label                           object
label                            int64
Length: 80, dtype: object
   Destination Port  Flow Duration  Total Fwd Packets  Total Backward Packets  \
0             54865              3                  2                       0   
1             55054            109                  1                       1   
2             55055             52                  1                       1   
3             46236             34                  1                       1   
4             54863              3                  2                       0   

   Total Length of Fwd Packets  Total Length